# Notebook 03 — Pipeline Dry Run

End-to-end dry run of **Layer 1 + Layer 2** on 5 Tier 1 prompts. No 3D generation. Validates that graph retrieval → scene spec generation works before committing GPU time.

**Set your `ANTHROPIC_API_KEY` in Cell 1 before running.**

In [1]:
import os, sys
from pathlib import Path

try:
    import google.colab  # type: ignore
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if not Path("pipeline").exists():
    if Path("../pipeline").exists():
        os.chdir("..")
    elif Path("ifc-graphrag-dt/pipeline").exists():
        os.chdir("ifc-graphrag-dt")
    elif IS_COLAB:
        !git clone https://github.com/aiwithprashant/ifc-graphrag-dt.git
        !bash ifc-graphrag-dt/colab_setup.sh
        os.chdir("ifc-graphrag-dt")
    else:
        raise RuntimeError("Run this notebook from the ifc-graphrag-dt repository root.")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"Working directory: {os.getcwd()}")

SPEC_PROVIDER = "anthropic" if os.environ.get("ANTHROPIC_API_KEY") else "deterministic"
print(f"Layer 2 provider: {SPEC_PROVIDER}")

import json
from pathlib import Path
from pipeline.layer1_retriever.ifc_graph_builder import IFCGraphBuilder
from pipeline.layer1_retriever.graph_embedder import GraphEmbedder
from pipeline.layer1_retriever.khop_traversal import KHopTraversal
from pipeline.layer2_spec_gen.scene_spec_generator import SceneSpecGenerator
from evaluation.stage_a.spec_evaluator import StageAEvaluator
from benchmark.dtah_bench import DTAHBench

IFC_PATH    = 'benchmark/ifc_reference_models/duplex.ifc'
GRAPH_CACHE = 'outputs/graphs/ifc_graph.json'
EMB_CACHE   = 'outputs/embedders/graph_embedder'
SPEC_OUT    = 'outputs/specs'
os.makedirs(SPEC_OUT, exist_ok=True)
os.makedirs('outputs/scores', exist_ok=True)
print('Imports OK')

Working directory: E:\PhD\Journal-2\Implementation\ifc-graphrag-dt
Layer 2 provider: deterministic


Imports OK


In [2]:
from pipeline.ifc_assets import ensure_duplex_ifc

DUPLEX_PATH = ensure_duplex_ifc()
IFC_PATH = str(DUPLEX_PATH)
print(f"IFC file ready: {DUPLEX_PATH} ({DUPLEX_PATH.stat().st_size / 1024:.1f} KB)")

if Path(GRAPH_CACHE).exists():
    G = IFCGraphBuilder.load_graph(GRAPH_CACHE)
else:
    builder = IFCGraphBuilder(IFC_PATH)
    G = builder.build()
    builder.save_graph(GRAPH_CACHE)

if Path(f'{EMB_CACHE}/faiss.index').exists():
    embedder = GraphEmbedder.load(EMB_CACHE)
else:
    print('Building embedder (2-5 min on CPU)...')
    embedder = GraphEmbedder(model_name='sentence-transformers/all-MiniLM-L6-v2')
    embedder.fit(G)
    embedder.save(EMB_CACHE)

print(f'Graph: {G.number_of_nodes()} nodes | Embedder: {len(embedder._node_ids)} nodes')

IFC file ready: benchmark\ifc_reference_models\duplex.ifc (2325.0 KB)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Graph: 296 nodes | Embedder: 296 nodes


In [3]:
# ── Cell 3: Initialise Layer 2 spec generator ─────────────────────────────────
spec_config = {
    'llm_provider': SPEC_PROVIDER,
    'model':        'claude-sonnet-4-20250514',
    'max_tokens':   2048,
    'temperature':  0.0,
    'output_dir':   SPEC_OUT,
}
spec_gen  = SceneSpecGenerator(spec_config)
evaluator = StageAEvaluator()
print('SceneSpecGenerator initialised with Anthropic Claude.')

SceneSpecGenerator initialised with Anthropic Claude.


In [4]:
# ── Cell 4: Dry run — 5 Tier 1 prompts ───────────────────────────────────────
bench   = DTAHBench(pilot_mode=True)
tier1   = bench.load_tier(1)[:5]
results = []

for p in tier1:
    pid, prompt, tier, domain = p['id'], p['prompt'], p['tier'], p.get('domain','MEP')
    print(f'\n[{pid}] {prompt}')

    # Layer 1
    seeds    = embedder.retrieve_seeds(prompt, top_k=5)
    seed_ids = [s['node_id'] for s in seeds]
    trav     = KHopTraversal(G, max_depth=1, bidirectional=True)  # Tier 1 → depth=1
    sg       = trav.traverse(seed_ids).subgraph
    print(f'  Layer 1: {sg.number_of_nodes()} nodes, {sg.number_of_edges()} edges')

    # Layer 2
    spec = spec_gen.generate(prompt=prompt, subgraph=sg,
                             prompt_id=pid, tier=tier, domain=domain)
    print(f'  Layer 2: {len(spec.get("entities",[]))} entities, '
          f'{len(spec.get("relations",[]))} relations')
    print(f'  SD prompt: {spec.get("generation_prompt","")[:80]}...')

    results.append({'id': pid, 'prompt': prompt, 'spec': spec,
                    'nodes': sg.number_of_nodes(), 'edges': sg.number_of_edges()})

print(f'\n✓ Dry run complete: {len(results)} prompts processed')


[T1-MEP-001] Generate a centrifugal pump
  Layer 1: 11 nodes, 7 edges
  Layer 2: 11 entities, 7 relations
  SD prompt: Generate a centrifugal pump. Technical 3D scene containing IfcBuildingStorey, If...

[T1-MEP-002] Generate a steel globe valve
  Layer 1: 7 nodes, 5 edges
  Layer 2: 7 entities, 5 relations
  SD prompt: Generate a steel globe valve. Technical 3D scene containing IfcBeam, IfcBuilding...

[T1-MEP-003] Generate an axial HVAC fan
  Layer 1: 10 nodes, 15 edges
  Layer 2: 10 entities, 15 relations
  SD prompt: Generate an axial HVAC fan. Technical 3D scene containing IfcBuildingStorey, Ifc...

[T1-MEP-004] Generate a pipe segment made of galvanised steel
  Layer 1: 7 nodes, 5 edges
  Layer 2: 7 entities, 5 relations
  SD prompt: Generate a pipe segment made of galvanised steel. Technical 3D scene containing ...

[T1-MEP-005] Generate a butterfly valve
  Layer 1: 10 nodes, 9 edges
  Layer 2: 10 entities, 9 relations
  SD prompt: Generate a butterfly valve. Technical 3D scene

In [5]:
# ── Cell 5: Inspect first spec ────────────────────────────────────────────────
print(f'Spec for: {results[0]["id"]} — {results[0]["prompt"]}')
print('='*60)
display_spec = {k: v for k, v in results[0]['spec'].items() if k != 'generation_prompt'}
print(json.dumps(display_spec, indent=2))

Spec for: T1-MEP-001 — Generate a centrifugal pump
{
  "entities": [
    {
      "id": "entity_0",
      "ifc_type": "IfcSpace",
      "name": "B103",
      "position": [
        0.0,
        0.0,
        0.0
      ],
      "attributes": {
        "LongName": "Kitchen",
        "CompositionType": "ELEMENT",
        "InteriorOrExteriorSpace": "INTERNAL"
      }
    },
    {
      "id": "entity_1",
      "ifc_type": "IfcSpace",
      "name": "A103",
      "position": [
        2.0,
        0.0,
        0.0
      ],
      "attributes": {
        "LongName": "Kitchen",
        "CompositionType": "ELEMENT",
        "InteriorOrExteriorSpace": "INTERNAL"
      }
    },
    {
      "id": "entity_2",
      "ifc_type": "IfcFurnishingElement",
      "name": "M_Counter Top w Sink Hole:600mm Depth:600mm Depth:162490",
      "position": [
        4.0,
        0.0,
        0.0
      ],
      "attributes": {
        "ObjectType": "600mm Depth",
        "Tag": "162490"
      }
    },
    {
      "id": 

In [6]:
# ── Cell 6: Stage A self-consistency check ────────────────────────────────────
print('Stage A self-consistency (pred vs self — should be ~1.0):')
print(f'{"ID":<15} {"E":>6} {"R":>6} {"A":>6} {"Total":>8}')
print('-' * 45)
for r in results:
    s = evaluator.evaluate(r['spec'], r['spec'], prompt_id=r['id'])
    print(f'{r["id"]:<15} {s.entity_score:>6.3f} {s.relation_score:>6.3f} '
          f'{s.attribute_score:>6.3f} {s.total:>8.3f}')
print('\nNote: Real Stage A scores require annotated ground-truth specs.')

Stage A self-consistency (pred vs self — should be ~1.0):
ID                   E      R      A    Total
---------------------------------------------
T1-MEP-001       1.000  1.000  1.000    1.000
T1-MEP-002       1.000  1.000  1.000    1.000
T1-MEP-003       1.000  1.000  1.000    1.000
T1-MEP-004       1.000  1.000  1.000    1.000
T1-MEP-005       1.000  1.000  1.000    1.000

Note: Real Stage A scores require annotated ground-truth specs.


In [7]:
# ── Cell 7: Save dry run summary ─────────────────────────────────────────────
summary = [{'id': r['id'], 'prompt': r['prompt'],
            'subgraph_nodes': r['nodes'], 'subgraph_edges': r['edges'],
            'spec_entities': len(r['spec'].get('entities',[])),
            'spec_relations': len(r['spec'].get('relations',[])),
            'spec_constraints': len(r['spec'].get('constraints',[])),
           } for r in results]

out = 'outputs/scores/dry_run_summary.json'
with open(out, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Summary saved: {out}')
print(json.dumps(summary, indent=2))
print('\nDry run complete. Proceed to Notebook 04 (GPU required).')

Summary saved: outputs/scores/dry_run_summary.json
[
  {
    "id": "T1-MEP-001",
    "prompt": "Generate a centrifugal pump",
    "subgraph_nodes": 11,
    "subgraph_edges": 7,
    "spec_entities": 11,
    "spec_relations": 7,
    "spec_constraints": 7
  },
  {
    "id": "T1-MEP-002",
    "prompt": "Generate a steel globe valve",
    "subgraph_nodes": 7,
    "subgraph_edges": 5,
    "spec_entities": 7,
    "spec_relations": 5,
    "spec_constraints": 5
  },
  {
    "id": "T1-MEP-003",
    "prompt": "Generate an axial HVAC fan",
    "subgraph_nodes": 10,
    "subgraph_edges": 15,
    "spec_entities": 10,
    "spec_relations": 15,
    "spec_constraints": 15
  },
  {
    "id": "T1-MEP-004",
    "prompt": "Generate a pipe segment made of galvanised steel",
    "subgraph_nodes": 7,
    "subgraph_edges": 5,
    "spec_entities": 7,
    "spec_relations": 5,
    "spec_constraints": 5
  },
  {
    "id": "T1-MEP-005",
    "prompt": "Generate a butterfly valve",
    "subgraph_nodes": 10,
    "subg